<a href="https://colab.research.google.com/github/whyaskidk91-web/saturnGPT_v2/blob/main/RUN_ME.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#[ALL] IMPORT + TOKENIZER + TRANSFORMER + LOAD + RUN
# -    thos code runs the model in one click with weigjts from HF

import torch,re,math as m,os
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from huggingface_hub import hf_hub_download
dev = "cuda" if torch.cuda.is_available() else "cpu"
END = "<|endoftext|>"
pat = re.compile(r"<\|endoftext\|>|'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?[^\w\s<]+| ?<+|\s+")
ctx = 768 #context window
print(dev,ctx)

# TOKENIZER FUNCTIONS
def get_pairs(ids):
  counts = {}
  for pair in zip(ids,ids[1:]):
    if pair in counts:
      counts[pair] += 1
    else:
      counts[pair] = 1
  return counts

def merge(ids,pair,new_id):
  out = []
  i = 0
  while i < len(ids):
    if (i+1) < len(ids) and (ids[i],ids[i + 1]) == pair:
      out.append(new_id)
      i += 2
    else:
      out.append(ids[i])
      i += 1
  return out

def clean(s): return bad.sub('',s)

cache = {}
def encode_word(w):
  if w in cache: return cache[w] #every word gets encoded only once
  ids = [stoi[c] for c in w]
  while len(ids) > 1:
    pairs = get_pairs(ids)
    best = min(pairs,key = lambda p: merges.get(p, float('inf'))) #merges in order
    if best not in merges: break
    ids = merge(ids,best,merges[best])
  if len(cache) < 3_000_000: cache[w] = ids
  return ids

def encode(text):
  num_text = []
  for w in pat.findall(clean(text)): #splits text into words,punct. and spaces
    num_text += encode_word(w)
  return num_text

def decode(ids):
  return ''.join(vocab[i] for i in ids)

# LOAD BUILT TOKENS
path = hf_hub_download(repo_id="3V4N3SC3NC3/saturnGPT_v2", filename="tok_120M.pt")
loaded = torch.load(path)
chars = loaded['characters'];merges = loaded['merges']
stoi = {c: a for a,c in enumerate(chars)}
itos = {a: c for a,c in enumerate(chars)}
bad = re.compile('[^' + re.escape(''.join(chars)) + ']')
vocab = {i: itos[i] for i in range(len(chars))}
for (a,b),new_id in merges.items():
  vocab[new_id] = vocab[a] + vocab[b]

end_id = len(chars) + len(merges)
vocab[end_id] = END
k = end_id + 1
cache = {END: [end_id]}
print("- Tokens loaded")

# TRANSFORMER
d = 768
nh = 12
it = 64
block_num = 14

class MultiHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.hs = d // nh
    self.proj = nn.Linear(d,d)
    self.query = nn.Linear(d,d,bias=False)
    self.key = nn.Linear(d,d,bias=False)
    self.value = nn.Linear(d,d,bias=False)

  def forward(self,x):
    B,T,_ = x.shape
    q = self.query(x).view(B,T,nh,self.hs).transpose(1,2)
    kk = self.key(x).view(B,T,nh,self.hs).transpose(1,2)
    v = self.value(x).view(B,T,nh,self.hs).transpose(1,2)
    out = F.scaled_dot_product_attention(q,kk,v,is_causal = True)
    out = out.transpose(1,2).contiguous().view(B,T,d)
    return self.proj(out)

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
    nn.Linear(d,d*4),
    nn.GELU(),
    nn.Linear(d*4,d),
    )

  def forward(self,x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self):
    super().__init__()
    self.ln1 = nn.LayerNorm(d)
    self.att = MultiHead()
    self.ln2 = nn.LayerNorm(d)
    self.ff = FeedForward()

  def forward(self,x):
    x = x + self.att(self.ln1(x))
    x = x + self.ff(self.ln2(x))
    return x

class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.tok = nn.Embedding(k,d)
    self.pos = nn.Embedding(ctx,d)
    self.blocks = nn.Sequential(*[Block() for _ in range(block_num)])
    self.ln = nn.LayerNorm(d)
    self.out = nn.Linear(d,k)

  def forward(self,x):
    B,T = x.shape
    x = self.tok(x) + self.pos(torch.arange(T,device=dev))
    x = self.blocks(x)
    x = self.ln(x)
    return self.out(x)

# LOAD FULL MODEL
path = hf_hub_download(repo_id="3V4N3SC3NC3/saturnGPT_v2", filename="chat_model_120M.pt")
model = Net().to(dev)
model.load_state_dict(torch.load(path,map_location=torch.device(dev)))
print("- Model loaded \n")
del path

# RUN FULL MODEL
model.eval()
while True:
  Temp = 0.7
  msg = input("You: ")
  if msg == "q": break
  ids = encode("User: " + clean(msg) + "\nAssistant:")
  out = []
  with torch.no_grad():
    for _ in range(200):
      x = torch.tensor([ids[-ctx:]],device=dev)
      probs = F.softmax(model(x)[:,-1,:] / Temp,dim=1)[0]
      t = torch.multinomial(probs,1).item()
      if t == end_id: break
      ids.append(t);out.append(t)
  print("Bot:",decode(out).strip())
